# Pipeline 1: 方面类别检测模型

本Notebook完成以下任务：
1. 加载预处理后的数据
2. 微调DistilBERT模型进行多标签分类（5个方面类别）
3. 评估模型性能
4. 保存并上传到Hugging Face

## 2.1 安装和导入必要的库

In [ ]:
# 安装必要的库
!pip install -q transformers datasets accelerate evaluate scikit-learn huggingface_hub

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
from torch import nn
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    TrainingArguments, 
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset, DatasetDict
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from huggingface_hub import notebook_login, HfApi
import os

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

# 检查GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")
print(f"✓ 库导入成功")

## 2.2 加载数据

In [ ]:
# 定义方面类别
ASPECT_CATEGORIES = ['food', 'service', 'price', 'ambience', 'anecdotes/miscellaneous']
NUM_LABELS = len(ASPECT_CATEGORIES)

# 加载数据
with open('GroupXX_Dataset_files/train_aspect_detection.json', 'r', encoding='utf-8') as f:
    train_data = json.load(f)

with open('GroupXX_Dataset_files/trial_aspect_detection.json', 'r', encoding='utf-8') as f:
    val_data = json.load(f)

print(f"训练集: {len(train_data)} 条")
print(f"验证集: {len(val_data)} 条")
print(f"方面类别: {ASPECT_CATEGORIES}")

In [ ]:
# 转换为Hugging Face Dataset格式
def prepare_dataset(data):
    """准备数据集，转换为Hugging Face格式"""
    texts = [item['text'] for item in data]
    labels = [item['labels'] for item in data]
    
    return Dataset.from_dict({
        'text': texts,
        'labels': labels
    })

train_dataset = prepare_dataset(train_data)
val_dataset = prepare_dataset(val_data)

print(f"\n训练集示例:")
print(train_dataset[0])

## 2.3 加载预训练模型和Tokenizer

In [ ]:
# 选择基础模型（推荐使用DistilBERT，轻量且效果好）
MODEL_NAME = "distilbert-base-uncased"  # 可选: bert-base-uncased, roberta-base

# 加载Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 加载模型（多标签分类）
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification"
).to(device)

print(f"✓ 模型 {MODEL_NAME} 加载成功")
print(f"  - 类别数: {NUM_LABELS}")
print(f"  - 问题类型: 多标签分类")

## 2.4 数据预处理（Tokenization）

In [ ]:
# 定义tokenize函数
def tokenize_function(examples):
    """对文本进行tokenization"""
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

# 应用tokenization
train_tokenized = train_dataset.map(tokenize_function, batched=True)
val_tokenized = val_dataset.map(tokenize_function, batched=True)

# 转换为torch tensor格式
train_tokenized.set_format(
    type="torch", 
    columns=["input_ids", "attention_mask", "labels"]
)
val_tokenized.set_format(
    type="torch", 
    columns=["input_ids", "attention_mask", "labels"]
)

print("✓ Tokenization完成")
print(f"训练集大小: {len(train_tokenized)}")
print(f"验证集大小: {len(val_tokenized)}")

## 2.5 定义评估指标

In [ ]:
def compute_metrics(eval_pred):
    """
    计算多标签分类的评估指标
    """
    logits, labels = eval_pred
    
    # 应用sigmoid并设置阈值0.5
    predictions = (torch.sigmoid(torch.tensor(logits)) > 0.5).int().numpy()
    labels = labels.astype(int)
    
    # 计算指标
    f1_micro = f1_score(labels, predictions, average='micro')
    f1_macro = f1_score(labels, predictions, average='macro')
    precision_micro = precision_score(labels, predictions, average='micro', zero_division=0)
    recall_micro = recall_score(labels, predictions, average='micro', zero_division=0)
    
    return {
        'f1_micro': f1_micro,
        'f1_macro': f1_macro,
        'precision_micro': precision_micro,
        'recall_micro': recall_micro
    }

print("✓ 评估指标函数定义完成")

## 2.6 设置训练参数

In [ ]:
# 定义训练参数
training_args = TrainingArguments(
    output_dir='./results_pipeline1',
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    greater_is_better=True,
    logging_dir='./logs_pipeline1',
    logging_steps=50,
    save_total_limit=2,
    push_to_hub=False,  # 训练完成后再手动上传
)

# 创建Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("✓ Trainer创建完成")
print(f"  - 学习率: {training_args.learning_rate}")
print(f"  - Batch size: {training_args.per_device_train_batch_size}")
print(f"  - 训练轮数: {training_args.num_train_epochs}")

## 2.7 开始训练

In [ ]:
# 开始训练
print("开始训练模型...")
trainer.train()

## 2.8 模型评估

In [ ]:
# 在验证集上进行评估
print("\n在验证集上评估...")
eval_results = trainer.evaluate()

print("\n===== 验证集评估结果 =====")
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}")

In [ ]:
# 获取详细分类报告
def get_detailed_report(trainer, dataset):
    """获取每个类别的详细评估报告"""
    predictions = trainer.predict(dataset)
    logits = predictions.predictions
    labels = predictions.label_ids
    
    # 应用sigmoid和阈值
    preds = (torch.sigmoid(torch.tensor(logits)) > 0.5).int().numpy()
    
    # 打印分类报告
    print("\n===== 每个类别的性能 =====")
    print(classification_report(
        labels, 
        preds, 
        target_names=ASPECT_CATEGORIES,
        zero_division=0
    ))
    
    return preds, labels

predictions, labels = get_detailed_report(trainer, val_tokenized)

## 2.9 保存模型

In [ ]:
# 保存模型到本地
MODEL_SAVE_PATH = 'GroupXX_Dataset_files/Fine-tuned_Model_files/aspect_detection_model'

# 创建目录
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

# 保存模型和tokenizer
trainer.save_model(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)

# 保存标签映射
label_config = {
    "id2label": {str(i): label for i, label in enumerate(ASPECT_CATEGORIES)},
    "label2id": {label: i for i, label in enumerate(ASPECT_CATEGORIES)}
}

with open(f"{MODEL_SAVE_PATH}/label_config.json", 'w', encoding='utf-8') as f:
    json.dump(label_config, f, indent=2)

print(f"✓ 模型已保存到: {MODEL_SAVE_PATH}")

## 2.10 测试模型推理

In [ ]:
# 测试模型推理
def predict_aspects(text, model, tokenizer):
    """预测文本中提到的方面类别"""
    inputs = tokenizer(
        text, 
        return_tensors="pt", 
        truncation=True, 
        padding=True,
        max_length=128
    ).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.sigmoid(logits).cpu().numpy()[0]
    
    # 获取预测结果（阈值0.5）
    predictions = (probs > 0.5).astype(int)
    
    # 构建结果
    detected_aspects = [
        ASPECT_CATEGORIES[i] 
        for i, pred in enumerate(predictions) 
        if pred == 1
    ]
    
    return {
        'aspects': detected_aspects,
        'probabilities': {
            ASPECT_CATEGORIES[i]: float(probs[i]) 
            for i in range(len(ASPECT_CATEGORIES))
        }
    }

# 测试示例
test_texts = [
    "The food was delicious but the service was slow and the price was too high.",
    "Great atmosphere and friendly staff!",
    "The pasta was amazing and reasonably priced."
]

print("===== 模型推理测试 =====\n")
for text in test_texts:
    result = predict_aspects(text, model, tokenizer)
    print(f"文本: {text}")
    print(f"检测到的方面: {result['aspects']}")
    print(f"概率: {result['probabilities']}")
    print("-" * 80)

## 2.11 上传到Hugging Face Hub

In [ ]:
# 登录Hugging Face（需要在Colab中输入token）
# notebook_login()

In [ ]:
# 上传到Hugging Face Hub
# 请替换为你的Hugging Face用户名和模型名称
HUB_MODEL_NAME = "your-username/absa-aspect-detection"

# 推送模型到Hub
# trainer.push_to_hub(HUB_MODEL_NAME)
# tokenizer.push_to_hub(HUB_MODEL_NAME)

print(f"模型上传命令（请在本地执行）:")
print(f"  huggingface-cli login")
print(f"  transformers-cli upload {MODEL_SAVE_PATH}")
print(f"\n或使用代码: trainer.push_to_hub('{HUB_MODEL_NAME}')")

## 2.12 记录实验结果

In [ ]:
# 记录实验结果
experiment_results = {
    "task": "Pipeline 1 - 方面类别检测",
    "base_model": MODEL_NAME,
    "num_labels": NUM_LABELS,
    "label_names": ASPECT_CATEGORIES,
    "hyperparameters": {
        "learning_rate": training_args.learning_rate,
        "batch_size": training_args.per_device_train_batch_size,
        "num_epochs": training_args.num_train_epochs,
        "weight_decay": training_args.weight_decay,
        "max_length": 128
    },
    "dataset": {
        "train_size": len(train_data),
        "val_size": len(val_data)
    },
    "evaluation_results": {
        "f1_micro": eval_results.get('eval_f1_micro', 0),
        "f1_macro": eval_results.get('eval_f1_macro', 0),
        "precision_micro": eval_results.get('eval_precision_micro', 0),
        "recall_micro": eval_results.get('eval_recall_micro', 0)
    },
    "model_path": MODEL_SAVE_PATH,
    "hub_model_name": HUB_MODEL_NAME
}

# 保存实验结果
with open('GroupXX_Dataset_files/experiment_pipeline1.json', 'w', encoding='utf-8') as f:
    json.dump(experiment_results, f, indent=2, ensure_ascii=False)

print("===== Pipeline 1 实验结果 =====")
print(json.dumps(experiment_results, indent=2, ensure_ascii=False))

print("\n✓ Pipeline 1 完成！")